In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn requests

In [ ]:
import pandas as pd
print("pandas works:", pd.__version__)

In [ ]:
import re
from pathlib import Path
import pandas as pd

# Make folders
for p in ["data/raw", "data/processed", "reports/figures", "src"]:
    Path(p).mkdir(parents=True, exist_ok=True)

# Download both datasets from opendata.com.pk
urls = {
    "wheat_production": "https://opendata.com.pk/datastore/dump/8f805abc-56c7-40d1-a2c1-708f31157479?bom=True",
    "wheat_area": "https://opendata.com.pk/datastore/dump/9b9f1869-834d-4fda-b9fc-5b6b73ac41b4?bom=True",
}
for name, url in urls.items():
    pd.read_csv(url, encoding="utf-8-sig").to_csv(f"data/raw/{name}.csv", index=False)
    print("Downloaded", name)

def load_clean(path, value_name):
    raw = pd.read_csv(path)
    raw = raw.drop(columns=[c for c in raw.columns if c == "_id"])
    raw = raw.rename(columns={raw.columns[0]: "district"})
    raw["district"] = raw["district"].astype(str).str.strip()
    drop_mask = raw["district"].str.contains(
        r"\bdiv(?:ision)?\b|^punjab$|total|^islamabad", case=False, regex=True
    )
    df = raw.loc[~drop_mask].copy()
    year_cols = [c for c in df.columns if re.match(r"\d{4}-\d{2}", str(c))]
    for c in year_cols:
        df[c] = pd.to_numeric(
            df[c].astype(str).str.replace(",", "", regex=False).str.strip(),
            errors="coerce",
        )
    long = df.melt(id_vars="district", value_vars=year_cols,
                   var_name="season", value_name=value_name)
    long["harvest_year"] = long["season"].str[:4].astype(int) + 1
    return long

prod = load_clean("data/raw/wheat_production.csv", "production_1000t")
area = load_clean("data/raw/wheat_area.csv", "area_1000ha")

panel = prod.merge(area, on=["district", "season", "harvest_year"], how="inner")
panel["yield_kg_ha"] = panel["production_1000t"] / panel["area_1000ha"] * 1000
panel.to_csv("data/processed/wheat_panel.csv", index=False)

print("Panel shape:", panel.shape, "(expect 468 rows)")
print("Districts:", panel["district"].nunique(), "(expect 36)")
print("Missing yield:", panel["yield_kg_ha"].isna().sum())
print(panel["yield_kg_ha"].describe().round(0))
display(panel.head(10))

In [ ]:
panel = pd.read_csv("data/processed/wheat_panel.csv")

# Lowest and highest yields: look for anything suspicious
print(panel.nsmallest(5, "yield_kg_ha")[["district", "harvest_year", "yield_kg_ha"]])
print(panel.nlargest(5, "yield_kg_ha")[["district", "harvest_year", "yield_kg_ha"]])

# Average yield per year
print(panel.groupby("harvest_year")["yield_kg_ha"].mean().round(0))

In [ ]:
import numpy as np

panel = pd.read_csv("data/processed/wheat_panel.csv")

# 1. Look at Kasur across all years
print(panel[panel["district"] == "Kasur"][
    ["harvest_year", "production_1000t", "area_1000ha", "yield_kg_ha"]
].round(0))

# 2. Mark the impossible value as missing, and keep a flag so it is documented
panel["yield_flag"] = ""
mask = (panel["district"] == "Kasur") & (panel["harvest_year"] == 2010)
panel.loc[mask, "yield_flag"] = "suspect_outlier"
panel.loc[mask, "yield_kg_ha"] = np.nan

# 3. Save the cleaned version
panel.to_csv("data/processed/wheat_panel_clean.csv", index=False)
print("\nMissing yield values now:", panel["yield_kg_ha"].isna().sum(), "(expect 3)")
print("Max yield now:", round(panel["yield_kg_ha"].max()))

In [ ]:
import time
import requests
import pandas as pd
from pathlib import Path

coords = {
    "Bahawalnagar": (29.9987, 73.2536), "Bahawalpur": (29.3956, 71.6836),
    "Rahim Yar Khan": (28.4202, 70.2952), "D.G.Khan": (30.0489, 70.6455),
    "Layyah": (30.9693, 70.9428), "Muzaffargarh": (30.0703, 71.1933),
    "Rajanpur": (29.1044, 70.3250), "Faisalabad": (31.4504, 73.1350),
    "Chiniot": (31.7292, 72.9788), "Jhang": (31.2681, 72.3181),
    "T.T.Singh": (30.9709, 72.4826), "Gujranwala": (32.1877, 74.1945),
    "Gujrat": (32.5731, 74.0754), "Hafizabad": (32.0709, 73.6880),
    "Mandi Baha-ud-Din": (32.5742, 73.4912), "Narowal": (32.1019, 74.8730),
    "Sialkot": (32.4945, 74.5229), "Kasur": (31.1187, 74.4467),
    "Lahore": (31.5497, 74.3436), "Nankana Sahib": (31.4492, 73.7120),
    "Sheikhupura": (31.7167, 73.9850), "Khanewal": (30.3017, 71.9321),
    "Lodhran": (29.5339, 71.6324), "Multan": (30.1575, 71.5249),
    "Vehari": (30.0452, 72.3489), "Attock": (33.7660, 72.3609),
    "Chakwal": (32.9328, 72.8630), "Jhelum": (32.9425, 73.7257),
    "Rawalpindi": (33.5651, 73.0169), "Sahiwal": (30.6682, 73.1114),
    "Okara": (30.8138, 73.4534), "Pakpattan": (30.3431, 73.3894),
    "Bhakkar": (31.6082, 71.0854), "Khushab": (32.2953, 72.3489),
    "Mianwali": (32.5839, 71.5370), "Sargodha": (32.0836, 72.6711),
}

panel = pd.read_csv("data/processed/wheat_panel_clean.csv")
missing = set(panel["district"]) - set(coords)
print("Districts without coordinates:", missing)   # should be set()

params = "PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN"
rows = []
for i, (name, (lat, lon)) in enumerate(coords.items(), 1):
    url = ("https://power.larc.nasa.gov/api/temporal/monthly/point"
           f"?parameters={params}&community=AG&longitude={lon}&latitude={lat}"
           "&start=2007&end=2020&format=JSON")
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            data = r.json()["properties"]["parameter"]
            break
        except Exception as e:
            print("  retry", name, e)
            time.sleep(3)
    else:
        print("FAILED:", name)
        continue
    for var, series in data.items():
        for ym, val in series.items():
            year, month = int(ym[:4]), int(ym[4:])
            if month <= 12:                     # month 13 = annual average, skip
                rows.append((name, year, month, var, val))
    print(f"{i}/36 done: {name}")

weather = pd.DataFrame(rows, columns=["district", "year", "month", "variable", "value"])
weather["value"] = weather["value"].replace(-999, float("nan"))   # -999 means missing
Path("data/raw").mkdir(parents=True, exist_ok=True)
weather.to_csv("data/raw/weather_monthly_long.csv", index=False)
print("\nSaved", weather.shape, "| districts:", weather["district"].nunique())
display(weather.head())

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

weather = pd.read_csv("data/raw/weather_monthly_long.csv")
panel = pd.read_csv("data/processed/wheat_panel_clean.csv")

# 1. Assign each month to a wheat season (harvest year)
#    Nov-Dec belong to the NEXT year's harvest; Jan-Apr to the same year's
weather = weather[weather["month"].isin([11, 12, 1, 2, 3, 4])].copy()
weather["harvest_year"] = np.where(weather["month"] >= 11, weather["year"] + 1, weather["year"])

# 2. Label growth stages
stage_map = {11: "nd", 12: "nd", 1: "jf", 2: "jf", 3: "ma", 4: "ma"}
weather["stage"] = weather["month"].map(stage_map)

# 3. Rainfall is given in mm/day, so convert to mm per month
weather["days"] = pd.to_datetime(
    dict(year=weather["year"], month=weather["month"], day=1)
).dt.days_in_month
rain = weather["variable"] == "PRECTOTCORR"
weather.loc[rain, "value"] = weather.loc[rain, "value"] * weather.loc[rain, "days"]

# 4. Seasonal features (whole season)
agg = {"PRECTOTCORR": "sum", "T2M": "mean", "T2M_MAX": "mean",
       "T2M_MIN": "mean", "ALLSKY_SFC_SW_DWN": "mean"}
names = {"PRECTOTCORR": "rain_mm", "T2M": "temp_mean", "T2M_MAX": "temp_max",
         "T2M_MIN": "temp_min", "ALLSKY_SFC_SW_DWN": "solar"}

season = weather.pivot_table(index=["district", "harvest_year"],
                             columns="variable", values="value",
                             aggfunc=lambda s: s.sum() if s.name == "PRECTOTCORR" else s.mean())
season = season.rename(columns=names)
season.columns = [f"season_{c}" for c in season.columns]

# 5. Stage features: rainfall, max temp and min temp per growth stage
stage_feats = []
for var, short, fn in [("PRECTOTCORR", "rain", "sum"),
                       ("T2M_MAX", "tmax", "mean"),
                       ("T2M_MIN", "tmin", "mean")]:
    sub = weather[weather["variable"] == var]
    t = sub.pivot_table(index=["district", "harvest_year"], columns="stage",
                        values="value", aggfunc=fn)
    t.columns = [f"{short}_{c}" for c in t.columns]
    stage_feats.append(t)

features = season.join(stage_feats).reset_index()

# 6. Merge with yield
model_table = panel.merge(features, on=["district", "harvest_year"], how="left")
model_table.to_csv("data/processed/wheat_model_table.csv", index=False)

print("Model table shape:", model_table.shape)
print("Districts:", model_table["district"].nunique())
print("Missing weather values:", int(model_table[features.columns[2:]].isna().sum().sum()))
print("Rows with yield:", model_table["yield_kg_ha"].notna().sum(), "(expect 465)")

# 7. Quick look: which features move with yield?
corr = (model_table.drop(columns=["district", "season", "yield_flag"], errors="ignore")
        .corr(numeric_only=True)["yield_kg_ha"].drop("yield_kg_ha").sort_values())
print("\nCorrelation with yield:")
print(corr.round(2))
display(model_table.head())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("data/processed/wheat_model_table.csv")

# Fix: season rainfall = sum of the three stage totals
df["season_rain_mm"] = df["rain_nd"] + df["rain_jf"] + df["rain_ma"]

df = df.dropna(subset=["yield_kg_ha"]).reset_index(drop=True)

# Features: weather only, plus year as a trend (NO production or area)
features = [c for c in df.columns if c.startswith(("season_", "rain_", "tmax_", "tmin_"))]
features += ["harvest_year"]
print("Features used:", len(features))
print(features)

X, y = df[features], df["yield_kg_ha"]

models = {
    "Mean baseline": DummyRegressor(),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                           random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05,
                                                       max_iter=300, random_state=42),
}

schemes = {
    "Random 5-fold": (KFold(5, shuffle=True, random_state=42), None),
    "Leave-one-district-out": (LeaveOneGroupOut(), df["district"]),
    "Leave-one-year-out": (LeaveOneGroupOut(), df["harvest_year"]),
}

rows = []
for scheme, (splitter, groups) in schemes.items():
    for name, model in models.items():
        pred = np.zeros(len(df))
        for tr, te in splitter.split(X, y, groups):
            m = clone(model).fit(X.iloc[tr], y.iloc[tr])
            pred[te] = m.predict(X.iloc[te])
        rows.append({
            "scheme": scheme, "model": name,
            "RMSE": np.sqrt(mean_squared_error(y, pred)),
            "MAE": mean_absolute_error(y, pred),
            "R2": r2_score(y, pred),
        })
    print("done:", scheme)

results = pd.DataFrame(rows)
results.to_csv("reports/validation_results.csv", index=False)

order = list(schemes)
print("\nRMSE (kg/ha), lower is better:")
display(results.pivot(index="model", columns="scheme", values="RMSE")[order].round(0))
print("R2, higher is better:")
display(results.pivot(index="model", columns="scheme", values="R2")[order].round(2))

# Test of my hypothesis: correlations WITHIN each district over time
dm = df.groupby("district")[["yield_kg_ha"] + features].transform(lambda s: s - s.mean())
print("\nCorrelation with yield after removing district averages:")
print(dm.corr()["yield_kg_ha"].drop("yield_kg_ha").sort_values().round(2))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv("data/processed/wheat_model_table.csv")
df["season_rain_mm"] = df["rain_nd"] + df["rain_jf"] + df["rain_ma"]
df = df.dropna(subset=["yield_kg_ha"]).reset_index(drop=True)
features = [c for c in df.columns if c.startswith(("season_", "rain_", "tmax_", "tmin_"))] + ["harvest_year"]
no_year = [f for f in features if f != "harvest_year"]
y = df["yield_kg_ha"]

def rmse(pred):
    return float(np.sqrt(mean_squared_error(y, pred)))

def oof_rf(feats, splitter, groups=None):
    pred = np.zeros(len(df))
    for tr, te in splitter.split(df, y, groups):
        m = RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                                  random_state=42, n_jobs=-1)
        m.fit(df.iloc[tr][feats], y.iloc[tr])
        pred[te] = m.predict(df.iloc[te][feats])
    return pred

def district_mean(splitter, groups=None):
    pred = np.zeros(len(df))
    for tr, te in splitter.split(df, y, groups):
        means = df.iloc[tr].groupby("district")["yield_kg_ha"].mean()
        pred[te] = df.iloc[te]["district"].map(means).fillna(y.iloc[tr].mean()).values
    return pred

# Tests 1 and 2: district-average baseline, and with/without harvest_year
schemes = {
    "Random 5-fold": (KFold(5, shuffle=True, random_state=42), None),
    "Leave-one-year-out": (LeaveOneGroupOut(), df["harvest_year"]),
}
rows = []
for name, (sp, g) in schemes.items():
    dm = rmse(district_mean(sp, g))
    rf = rmse(oof_rf(features, sp, g))
    rf_ny = rmse(oof_rf(no_year, sp, g))
    rows.append({"scheme": name,
                 "RMSE district-mean": dm,
                 "RMSE RF": rf,
                 "RMSE RF no year": rf_ny,
                 "skill vs district-mean (RF)": 1 - rf**2 / dm**2,
                 "skill vs district-mean (RF no year)": 1 - rf_ny**2 / dm**2})
print("Skill > 0 means the model beats simply knowing each district's average.\n")
display(pd.DataFrame(rows).set_index("scheme").round(2))

g = df["district"]
print("Leave-one-district-out RMSE: with year",
      round(rmse(oof_rf(features, LeaveOneGroupOut(), g))),
      "| without year", round(rmse(oof_rf(no_year, LeaveOneGroupOut(), g))))

# Test 3: leave-one-region-out
regions = {
    "South": ["Bahawalnagar", "Bahawalpur", "Rahim Yar Khan", "D.G.Khan", "Layyah",
              "Muzaffargarh", "Rajanpur", "Khanewal", "Lodhran", "Multan", "Vehari"],
    "Central": ["Faisalabad", "Chiniot", "Jhang", "T.T.Singh", "Gujranwala", "Gujrat",
                "Hafizabad", "Mandi Baha-ud-Din", "Narowal", "Sialkot", "Kasur", "Lahore",
                "Nankana Sahib", "Sheikhupura", "Sahiwal", "Okara", "Pakpattan"],
    "Thal": ["Bhakkar", "Khushab", "Mianwali", "Sargodha"],
    "Pothohar": ["Attock", "Chakwal", "Jhelum", "Rawalpindi"],
}
region_of = {d: r for r, ds in regions.items() for d in ds}
df["region"] = df["district"].map(region_of)
print("\nUnassigned districts:", df.loc[df["region"].isna(), "district"].unique())

pred = oof_rf(features, LeaveOneGroupOut(), df["region"])
out = []
for r in regions:
    m = (df["region"] == r).values
    out.append({"held-out region": r,
                "districts": df.loc[m, "district"].nunique(),
                "RMSE model": np.sqrt(np.mean((y[m] - pred[m]) ** 2)),
                "RMSE mean-baseline": np.sqrt(np.mean((y[m] - y[~m].mean()) ** 2))})
display(pd.DataFrame(out).round(0))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("data/processed/wheat_model_table.csv")
df["season_rain_mm"] = df["rain_nd"] + df["rain_jf"] + df["rain_ma"]
df = df.dropna(subset=["yield_kg_ha"]).reset_index(drop=True)
wfeats = [c for c in df.columns if c.startswith(("season_", "rain_", "tmax_", "tmin_"))]
y = df["yield_kg_ha"]

def make_models():
    return {
        "Year trend only": (["harvest_year"], LinearRegression()),
        "Ridge: weather anomalies": (wfeats, make_pipeline(StandardScaler(), Ridge(alpha=10.0))),
        "Ridge: weather anomalies + year": (wfeats + ["harvest_year"],
                                            make_pipeline(StandardScaler(), Ridge(alpha=10.0))),
        "Random Forest: weather anomalies": (wfeats, RandomForestRegressor(
            n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)),
    }

preds = {"District mean (baseline)": np.zeros(len(df))}
for name in make_models():
    preds[name] = np.zeros(len(df))

for yr in sorted(df["harvest_year"].unique()):
    te = (df["harvest_year"] == yr).values
    train, test = df[~te], df[te]

    # District averages from TRAINING years only
    dm_y = train.groupby("district")["yield_kg_ha"].mean()
    dm_w = train.groupby("district")[wfeats].mean()
    base_te = test["district"].map(dm_y).values
    preds["District mean (baseline)"][te] = base_te

    # Anomalies: value minus the district's own training average
    Xtr = train[wfeats] - dm_w.loc[train["district"]].values
    Xte = test[wfeats] - dm_w.loc[test["district"]].values
    Xtr["harvest_year"] = train["harvest_year"].values
    Xte["harvest_year"] = test["harvest_year"].values
    ytr = train["yield_kg_ha"].values - train["district"].map(dm_y).values

    for name, (cols, model) in make_models().items():
        model.fit(Xtr[cols], ytr)
        preds[name][te] = base_te + model.predict(Xte[cols])

rows = []
base_rmse = np.sqrt(np.mean((y - preds["District mean (baseline)"]) ** 2))
for name, p in preds.items():
    rmse = np.sqrt(np.mean((y - p) ** 2))
    rows.append({"model": name, "RMSE": rmse, "MAE": np.mean(np.abs(y - p)),
                 "skill vs district mean": 1 - (rmse / base_rmse) ** 2})

res = pd.DataFrame(rows).set_index("model").sort_values("RMSE")
print("Leave-one-year-out, predicting each district's deviation from its own average")
print("Skill > 0: the model beats the plain district average\n")
display(res.round(2))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/processed/wheat_model_table.csv")
df["season_rain_mm"] = df["rain_nd"] + df["rain_jf"] + df["rain_ma"]
df = df.dropna(subset=["yield_kg_ha"]).reset_index(drop=True)
y = df["yield_kg_ha"].values
years = sorted(df["harvest_year"].unique())
wfeats = [c for c in df.columns if c.startswith(("season_", "rain_", "tmax_", "tmin_"))]

def district_mean():
    pred = np.zeros(len(df))
    for yr in years:
        te = (df["harvest_year"] == yr).values
        dm = df[~te].groupby("district")["yield_kg_ha"].mean()
        pred[te] = df[te]["district"].map(dm).values
    return pred

def loyo(weather_cols, alpha=10.0):
    """Leave-one-year-out. Predicts deviation from the district mean (training years only)."""
    pred = np.zeros(len(df))
    for yr in years:
        te = (df["harvest_year"] == yr).values
        train, test = df[~te], df[te]
        dm_y = train.groupby("district")["yield_kg_ha"].mean()
        base_te = test["district"].map(dm_y).values
        ytr = train["yield_kg_ha"].values - train["district"].map(dm_y).values
        Xtr = pd.DataFrame({"harvest_year": train["harvest_year"].values})
        Xte = pd.DataFrame({"harvest_year": test["harvest_year"].values})
        if weather_cols:
            dm_w = train.groupby("district")[weather_cols].mean()
            Xtr[weather_cols] = train[weather_cols].values - dm_w.loc[train["district"]].values
            Xte[weather_cols] = test[weather_cols].values - dm_w.loc[test["district"]].values
        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        model.fit(Xtr, ytr)
        pred[te] = base_te + model.predict(Xte)
    return pred

def rmse(p, m=None):
    m = slice(None) if m is None else m
    return float(np.sqrt(np.mean((y[m] - p[m]) ** 2)))

sets = {
    "Year trend only": [],
    "+ tmin_ma": ["tmin_ma"],
    "+ tmin_ma, tmax_ma": ["tmin_ma", "tmax_ma"],
    "+ tmin_ma, tmax_ma, rain_jf": ["tmin_ma", "tmax_ma", "rain_jf"],
    "+ all weather anomalies": wfeats,
}

base = district_mean()
year_only = loyo([])
masks = [(df["harvest_year"] == yr).values for yr in years]

rows = []
for name, cols in sets.items():
    p = loyo(cols)
    wins = np.nan if not cols else sum(rmse(p, m) < rmse(year_only, m) for m in masks)
    rows.append({"model": name,
                 "RMSE": rmse(p),
                 "skill vs district mean": 1 - (rmse(p) / rmse(base)) ** 2,
                 "years better than trend-only (of 13)": wins})

print("District mean RMSE:", round(rmse(base), 1))
display(pd.DataFrame(rows).set_index("model").round(2))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

df = pd.read_csv("data/processed/wheat_model_table.csv")
df = df.dropna(subset=["yield_kg_ha"]).reset_index(drop=True)
years = sorted(df["harvest_year"].unique())
MISCOVERAGE = 0.10   # target: 90% of true yields inside the interval

def fit_predict(train, test, cols):
    """District mean + Ridge on year trend and weather anomalies."""
    dm_y = train.groupby("district")["yield_kg_ha"].mean()
    base_tr = train["district"].map(dm_y).values
    base_te = test["district"].map(dm_y).fillna(train["yield_kg_ha"].mean()).values
    Xtr = pd.DataFrame({"harvest_year": train["harvest_year"].values})
    Xte = pd.DataFrame({"harvest_year": test["harvest_year"].values})
    if cols:
        dm_w = train.groupby("district")[cols].mean()
        w_tr = dm_w.reindex(train["district"]).values
        w_te = dm_w.reindex(test["district"]).values
        w_te = np.where(np.isnan(w_te), test[cols].values, w_te)
        Xtr[cols] = train[cols].values - w_tr
        Xte[cols] = test[cols].values - w_te
    ytr = train["yield_kg_ha"].values - base_tr
    model = make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(Xtr, ytr)
    return base_te + model.predict(Xte)

def conformal_q(res, miscov=MISCOVERAGE):
    res = np.sort(res[~np.isnan(res)])
    n = len(res)
    k = int(np.ceil((n + 1) * (1 - miscov)))
    return res[min(k, n) - 1]

def run(cols):
    out = []
    for T in years:
        te_mask = (df["harvest_year"] == T).values
        train, test = df[~te_mask], df[te_mask]
        pred = fit_predict(train, test, cols)
        ytrue = test["yield_kg_ha"].values

        # Blocked: residuals from leave-one-year-out WITHIN the training years
        res_b = []
        for C in sorted(train["harvest_year"].unique()):
            m = (train["harvest_year"] == C).values
            p = fit_predict(train[~m], train[m], cols)
            res_b.append(np.abs(train[m]["yield_kg_ha"].values - p))
        q_b = conformal_q(np.concatenate(res_b))

        # Naive: residuals from a random 5-fold split that ignores years
        res_n = np.zeros(len(train))
        for tr, ca in KFold(5, shuffle=True, random_state=0).split(train):
            p = fit_predict(train.iloc[tr], train.iloc[ca], cols)
            res_n[ca] = np.abs(train.iloc[ca]["yield_kg_ha"].values - p)
        q_n = conformal_q(res_n)

        for method, q in [("Naive (random-split)", q_n), ("Blocked (year-out)", q_b)]:
            out.append({"year": T, "method": method, "q": q,
                        "covered": np.mean(np.abs(ytrue - pred) <= q), "n": len(ytrue)})
    return pd.DataFrame(out)

specs = {"Year trend only": [], "Year + tmin_ma": ["tmin_ma"]}
summary, detail = [], {}
for name, cols in specs.items():
    r = run(cols)
    detail[name] = r
    for method, g in r.groupby("method"):
        summary.append({"model": name, "method": method,
                        "coverage": (g["covered"] * g["n"]).sum() / g["n"].sum(),
                        "worst-year coverage": g["covered"].min(),
                        "interval width (kg/ha)": 2 * g["q"].mean()})

print("Target coverage: 0.90. Each test year is completely held out.\n")
display(pd.DataFrame(summary).set_index(["model", "method"]).round(2))

print("Coverage by held-out year (Year + tmin_ma):")
display(detail["Year + tmin_ma"].pivot(index="year", columns="method", values="covered").round(2))

In [ ]:
def run_rows(cols):
    rows = []
    for T in years:
        te_mask = (df["harvest_year"] == T).values
        train, test = df[~te_mask], df[te_mask]
        pred = fit_predict(train, test, cols)
        res_b = []
        for C in sorted(train["harvest_year"].unique()):
            m = (train["harvest_year"] == C).values
            p = fit_predict(train[~m], train[m], cols)
            res_b.append(np.abs(train[m]["yield_kg_ha"].values - p))
        q = conformal_q(np.concatenate(res_b))
        err = np.abs(test["yield_kg_ha"].values - pred)
        rows.append(pd.DataFrame({"district": test["district"].values, "year": T,
                                  "abs_err": err, "q": q, "covered": err <= q}))
    return pd.concat(rows, ignore_index=True)

rows = run_rows(["tmin_ma"])

per_district = rows.groupby("district").agg(
    coverage=("covered", "mean"), mean_abs_error=("abs_err", "mean"),
    n_years=("year", "count")).reset_index()
per_district["yield_std"] = per_district["district"].map(
    df.groupby("district")["yield_kg_ha"].std())
per_district["variability"] = pd.qcut(
    per_district["yield_std"], 3, labels=["low", "medium", "high"])

print("Coverage by how variable a district's yields are (target 0.90):")
display(per_district.groupby("variability", observed=True).agg(
    districts=("district", "count"),
    coverage=("coverage", "mean"),
    mean_abs_error=("mean_abs_error", "mean"),
    yield_std=("yield_std", "mean")).round(2))

print("Five districts with the lowest coverage:")
display(per_district.sort_values("coverage").head(5)
        [["district", "coverage", "mean_abs_error", "yield_std"]].round(2))

Path("reports").mkdir(exist_ok=True)
per_district.to_csv("reports/conformal_by_district.csv", index=False)

In [ ]:
from pathlib import Path

Path("reports").mkdir(exist_ok=True)
per_district.to_csv("reports/conformal_by_district.csv", index=False)
print("Saved reports/conformal_by_district.csv")

In [ ]:
def run_adaptive(cols, shrink=0.5):
    out = []
    for T in years:
        te_mask = (df["harvest_year"] == T).values
        train, test = df[~te_mask], df[te_mask]
        pred = fit_predict(train, test, cols)
        ytrue = test["yield_kg_ha"].values

        # Calibration residuals: leave-one-year-out within the training years
        cal = []
        for C in sorted(train["harvest_year"].unique()):
            m = (train["harvest_year"] == C).values
            p = fit_predict(train[~m], train[m], cols)
            cal.append(pd.DataFrame({
                "district": train[m]["district"].values,
                "res": np.abs(train[m]["yield_kg_ha"].values - p)}))
        cal = pd.concat(cal, ignore_index=True)

        # Typical error size per district, shrunk toward the overall average
        glob = cal["res"].mean()
        sig = cal.groupby("district")["res"].mean()
        sig = shrink * glob + (1 - shrink) * sig
        cal["sigma"] = cal["district"].map(sig)

        q_norm = conformal_q((cal["res"] / cal["sigma"]).values)
        q_fixed = conformal_q(cal["res"].values)

        sig_te = test["district"].map(sig).fillna(glob).values
        err = np.abs(ytrue - pred)
        out.append(pd.DataFrame({
            "district": test["district"].values, "year": T,
            "fixed_cov": err <= q_fixed, "adapt_cov": err <= q_norm * sig_te,
            "fixed_w": 2 * q_fixed, "adapt_w": 2 * q_norm * sig_te}))
    return pd.concat(out, ignore_index=True)

res = run_adaptive(["tmin_ma"])
res["variability"] = res["district"].map(per_district.set_index("district")["variability"])

print("Overall (target coverage 0.90):")
display(pd.DataFrame(
    {"coverage": [res["fixed_cov"].mean(), res["adapt_cov"].mean()],
     "mean width (kg/ha)": [res["fixed_w"].mean(), res["adapt_w"].mean()]},
    index=["Fixed width", "Adaptive width"]).round(2))

g = res.groupby("variability", observed=True)
print("By district variability:")
display(pd.DataFrame({
    "fixed coverage": g["fixed_cov"].mean(), "adaptive coverage": g["adapt_cov"].mean(),
    "fixed width": g["fixed_w"].mean(), "adaptive width": g["adapt_w"].mean()}).round(2))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

Path("reports/figures").mkdir(parents=True, exist_ok=True)
df = pd.read_csv("data/processed/wheat_model_table.csv")

# Figure 1: yield over time
fig, ax = plt.subplots(figsize=(8, 4.5))
for d, g in df.groupby("district"):
    ax.plot(g["harvest_year"], g["yield_kg_ha"], color="grey", alpha=0.25, lw=0.8)
mean_by_year = df.groupby("harvest_year")["yield_kg_ha"].mean()
ax.plot(mean_by_year.index, mean_by_year.values, color="tab:green", lw=3,
        label="Average of all districts")
ax.set_xlabel("Harvest year")
ax.set_ylabel("Wheat yield (kg/ha)")
ax.set_title("Punjab wheat yield by district, 2008-2020")
ax.legend()
fig.tight_layout()
fig.savefig("reports/figures/fig1_yield_trend.png", dpi=200)
plt.show()

# Figure 2: leave-one-year-out error (values copied from our outputs)
rmse = {
    "Year trend + tmin_ma": 241, "Year trend + all weather anomalies": 247,
    "Year trend only": 251, "District average": 281,
    "Random Forest, weather anomalies": 324, "Ridge, weather anomalies": 334,
    "Random Forest, raw weather + year": 534,
}
s = pd.Series(rmse).sort_values(ascending=False)
colors = ["tab:red" if k == "District average" else "tab:blue" for k in s.index]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(s.index, s.values, color=colors)
for i, v in enumerate(s.values):
    ax.text(v + 5, i, str(v), va="center")
ax.set_xlabel("Prediction error, RMSE (kg/ha), lower is better")
ax.set_title("Leave-one-year-out error (red = district-average baseline)")
fig.tight_layout()
fig.savefig("reports/figures/fig2_model_comparison.png", dpi=200)
plt.show()

# Figure 3: interval coverage by district variability (copied from outputs)
groups = ["low", "medium", "high"]
fixed = [0.98, 0.88, 0.84]
adaptive = [0.95, 0.85, 0.87]
x = range(3)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([i - 0.2 for i in x], fixed, 0.4, label="Fixed-width intervals")
ax.bar([i + 0.2 for i in x], adaptive, 0.4, label="Adaptive-width intervals")
ax.axhline(0.90, color="black", ls="--", lw=1, label="Target 0.90")
ax.set_xticks(list(x))
ax.set_xticklabels(groups)
ax.set_xlabel("District yield variability")
ax.set_ylabel("Coverage")
ax.set_ylim(0.7, 1.02)
ax.set_title("Prediction-interval coverage by district variability")
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig("reports/figures/fig3_coverage.png", dpi=200)
plt.show()
print("Saved 3 figures in reports/figures/")

In [ ]:
import matplotlib.pyplot as plt

groups = ["low", "medium", "high"]
fixed, adaptive = [0.98, 0.88, 0.84], [0.95, 0.85, 0.87]
x = range(3)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([i - 0.2 for i in x], fixed, 0.4, label="Fixed-width intervals")
ax.bar([i + 0.2 for i in x], adaptive, 0.4, label="Adaptive-width intervals")
ax.axhline(0.90, color="black", ls="--", lw=1, label="Target 0.90")
ax.set_xticks(list(x))
ax.set_xticklabels(groups)
ax.set_xlabel("District yield variability")
ax.set_ylabel("Coverage")
ax.set_ylim(0.7, 1.12)
ax.set_yticks([0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0])
ax.set_title("Prediction-interval coverage by district variability")
ax.legend(loc="upper right", ncol=1, fontsize=8)
fig.tight_layout()
fig.savefig("reports/figures/fig3_coverage.png", dpi=200)
plt.show()
